In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
sns.set_theme()

import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv("Unemployment in India.csv")
df

In [ ]:
print("Rows and Columns:", df.shape)
df.info()

In [ ]:
df.columns = df.columns.str.strip()

df.rename(columns={
    "Estimated Unemployment Rate (%)":"Unemployment_Rate",
    "Estimated Employed":"Employed",
    "Estimated Labour Participation Rate (%)":"Labour_Participation_Rate"
}, inplace=True)

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)
df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
print("Average Unemployment Rate:")

print(round(df["Unemployment_Rate"].mean(),2),"%")

In [ ]:
plt.figure(figsize=(8,6))

sns.histplot(
    df["Unemployment_Rate"],
    kde=True
)

plt.title("Distribution of Unemployment Rate")

plt.show()

In [ ]:
state_rate = df.groupby("Region")["Unemployment_Rate"]\
               .mean()\
               .sort_values(ascending=False)

state_rate.head(10)

In [ ]:
plt.figure(figsize=(10,8))

state_rate.head(10).plot(
    kind="bar",
    color="steelblue"
)

plt.title("Top 10 States with Highest Average Unemployment")
plt.ylabel("Unemployment Rate (%)")

plt.show()

In [ ]:
monthly_trend = df.groupby("Date")["Unemployment_Rate"].mean()

plt.figure(figsize=(8,6))

monthly_trend.plot(
    marker="o"
)

plt.title("Monthly Unemployment Trend in India")
plt.ylabel("Unemployment Rate (%)")

plt.show()

In [ ]:
pre_covid = df[df["Date"] < "2020-03-01"]

covid = df[df["Date"] >= "2020-03-01"]

In [ ]:
pre_rate = pre_covid["Unemployment_Rate"].mean()

covid_rate = covid["Unemployment_Rate"].mean()

comparison = pd.DataFrame({
    "Period":["Pre-Covid","Covid"],
    "Rate":[pre_rate,covid_rate]
})

comparison

In [ ]:
plt.figure(figsize=(5,4))

sns.barplot(
    data=comparison,
    x="Period",
    y="Rate"
)

plt.title("Impact of Covid-19 on Unemployment")

plt.show()

In [ ]:
covid_monthly = covid.groupby("Date")["Unemployment_Rate"].mean()

plt.figure(figsize=(8,6))

covid_monthly.plot(
    color="red",
    marker="o"
)

plt.title("Unemployment Trend During Covid-19")

plt.ylabel("Rate (%)")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.boxplot(
    x="Area",
    y="Unemployment_Rate",
    data=df
)

plt.title("Urban vs Rural Unemployment")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    x="Labour_Participation_Rate",
    y="Unemployment_Rate",
    hue="Area",
    data=df
)

plt.title("Labour Participation vs Unemployment")

plt.show()

In [ ]:
corr = df[
    [
        "Unemployment_Rate",
        "Employed",
        "Labour_Participation_Rate"
    ]
].corr()

plt.figure(figsize=(6,4))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
df["Month"] = df["Date"].dt.month_name()

month_order = [
    'January','February','March',
    'April','May','June',
    'July','August','September',
    'October','November','December'
]

monthly_pattern = df.groupby("Month")["Unemployment_Rate"]\
                    .mean()\
                    .reindex(month_order)

monthly_pattern

In [ ]:
plt.figure(figsize=(8,4))
monthly_pattern.plot(
    kind="bar"
)
plt.title("Seasonal Pattern of Unemployment")
plt.ylabel("Average Rate (%)")

plt.show()

In [ ]:
employment = df.groupby("Region")["Employed"]\
               .mean()\
               .sort_values(ascending=False)\
               .head(10)

plt.figure(figsize=(10,4))
employment.plot(
    kind="bar"
)
plt.title("Top States by Employment")
plt.ylabel("Estimated Employed")

plt.show()

In [ ]:
!pip install plotly

In [ ]:
import plotly.express as px

fig = px.histogram(
    df.head(100),
    x='Date',
    y='Unemployment_Rate',
    color='Region',
    title='State-wise Unemployment Trends'

)

fig.show()

In [ ]:
pre = pre_covid.groupby('Region')['Unemployment_Rate'].mean()

post = covid.groupby('Region')['Unemployment_Rate'].mean()

impact = (post - pre).sort_values(ascending=False)

impact.head(10)
impact.head(10).plot(kind='bar')

In [ ]:
state_rank = df.groupby('Region')['Unemployment_Rate'].mean()

state_rank = state_rank.sort_values()

state_rank

In [ ]:
def category(rate):
    if rate < 8:
        return 'Low'
    elif rate < 15:
        return 'Moderate'
    else:
        return 'High'

ranking = state_rank.reset_index()

ranking['Category'] = ranking[
    'Unemployment_Rate'
].apply(category)
ranking

In [ ]:
from sklearn.linear_model import LinearRegression
monthly = df.groupby('Date')[
    'Unemployment_Rate'
].mean().reset_index()

monthly['Days'] = (
    monthly['Date']
    - monthly['Date'].min()
).dt.days
X = monthly[['Days']]
y = monthly['Unemployment_Rate']

model = LinearRegression()

model.fit(X,y)

In [ ]:
peak = df.loc[
    df['Unemployment_Rate'].idxmax()]
print(peak)

sns.lineplot(
    x='Date',
    y='Unemployment_Rate',
    data=df
)

plt.axvline(
    peak['Date'],
    color='red'
)

In [ ]:
sns.regplot(
    x='Labour_Participation_Rate',
    y='Unemployment_Rate',
    color='red',
    data=df
)

In [ ]:
import plotly.express as px

In [ ]:
!pip install ydata-profiling

In [ ]:
import pandas as pd
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="CodeAlpha Unemployment Analysis Report",
    explorative=True,
    correlations={
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
        "kendall": {"calculate": True},
        "phi_k": {"calculate": True}
    },
    missing_diagrams={
        "matrix": True,
        "heatmap": True,
        "dendrogram": True
    }
)

profile.to_file("CodeAlpha_Unemployment_Report.html")